# Week 01A — Machine Vision Systems and Digital Images

**MCTA 4364 Machine Vision** · Session 1 of Week 1 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W01_images_colour/W01A_vision_systems_images.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W01_images_colour/W01A_vision_systems_images.ipynb)

Welcome to Machine Vision! A machine-vision system lets a machine **see, measure and decide**: inspect bottles on a
production line, guide a robot to pick a part, read a serial number, or count products. This first session explains what
such a system is made of and what a **digital image** really is: a grid of numbers produced by *sampling* and *quantising* light.

### Learning outcomes
By the end of this session you will be able to:
1. Name the components of a machine-vision system and describe typical industrial applications.
2. Explain **sampling** (resolution) and **quantisation** (bit depth), and recognise **aliasing** and **false contours**.
3. Handle images as NumPy arrays: shape, data type, coordinates, regions of interest (ROIs).
4. Avoid classic pitfalls: `uint8` overflow, BGR vs RGB, slow Python loops, lossy JPEG compression.
5. Build a first inspection check: *is the part present?*

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–12 | 1 | What is a machine-vision system? |
| 12–25 | 2–3 | From light to numbers; images as arrays |
| 25–45 | 4–5 | Sampling, aliasing, quantisation (interactive) |
| 45–60 | 6–8 | Data types, basic operations, speed, file formats |
| 60–72 | 9 | Case study: part-presence inspection |
| 72–80 | 12 | Quiz and exit ticket |
| Home | 10–11 | Exercises with self-checks, webcam challenge |

**Prerequisites:** basic Python. Run **W00 Setup Check** first if you are working on your own laptop.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import skimage.data

from cvhelpers import SMOKE, check, interact, load_sample, pipeline_diagram, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

## 1. What is a machine-vision system?

**Computer vision** is the science of extracting information from images. **Machine vision** is its engineering application
in industry and robotics: a *system* that images a scene under controlled conditions and turns the image into a **decision
or measurement that drives a machine**. Reliability, speed and repeatability matter as much as accuracy.

In [ ]:
pipeline_diagram(["Part /\nscene", "Lighting", "Lens\n(optics)", "Image\nsensor", "Interface\n(USB3, GigE)",
                  "Processing\n(PC, edge GPU)", "Decision", "Actuator\n/ PLC"],
                 colors=["#f1f3f4", "#fce8b2", "#fce8b2", "#fce8b2", "#e8f0fe", "#ceead6", "#ceead6", "#fad2cf"],
                 notes=["", "Week 2", "Week 2", "Week 1-2", "", "Weeks 3-13", "", "reject, pick, sort"],
                 title="Figure 1.1 - Components of a machine-vision system", figsize=(18, 2.6))

| Application | Question the system answers | Example | Course weeks |
|---|---|---|---|
| **Presence / absence** | Is the part (cap, label, screw) there? | bottle-cap check | 1, 4 |
| **Gauging / measurement** | How big is it, in millimetres? | hole diameter, gap width | 2, 4 |
| **Inspection / defects** | Is it scratched, cracked, contaminated? | PCB, weld, fabric | 3, 11 |
| **Identification** | What is it / what does it say? | barcode, OCR, colour sorting | 1, 5, 13 |
| **Guidance / localisation** | Where is it, and how is it oriented? | robot pick-and-place | 5, 8, 12 |
| **Counting and tracking** | How many, and where are they going? | conveyor counting, traffic | 8, 10 |

> **Machine vision vs human inspection.** A camera never gets tired, measures to a fraction of a pixel and can inspect
> thousands of parts per minute, but it only sees what its **lighting, optics and algorithms** were designed to see.
> Most failed vision projects fail because of the image, not the algorithm.

**Watch (optional, 10 min):** Shree Nayar's free lecture series *First Principles of Computer Vision* (Columbia University) starts with an
excellent overview of what vision systems do: [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/).

## 2. From light to numbers

A camera turns a continuous pattern of light $f(x, y)$ into a finite grid of integers $I[r, c]$ in two steps:

1. **Sampling** (spatial): the sensor measures light only at a grid of **pixels**, spaced $\Delta$ apart. More pixels = higher **resolution**.
2. **Quantisation** (intensity): each measured brightness is rounded to one of $2^b$ levels. $b$ is the **bit depth**
   (8 bits = 256 grey levels).

$$ I[r, c] = Q\big(f(c\,\Delta,\; r\,\Delta)\big), \qquad Q(v) = \text{round}\left(\frac{v}{v_\text{max}}(2^b - 1)\right) $$

In [ ]:
t = np.linspace(0, 1, 1000)
signal = 0.5 + 0.4 * np.sin(2 * np.pi * 3 * t) + 0.08 * np.sin(2 * np.pi * 11 * t)
ts = np.linspace(0, 1, 25)
samples = np.interp(ts, t, signal)
levels = 8
quantised = np.round(samples * (levels - 1)) / (levels - 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8), sharey=True)
axes[0].plot(t, signal, lw=2); axes[0].set_title("continuous light intensity f(x)")
axes[1].plot(t, signal, color="#9aa0a6"); axes[1].stem(ts, samples, basefmt=" ")
axes[1].set_title("sampling: measure at 25 pixel positions")
axes[2].plot(t, signal, color="#9aa0a6")
axes[2].step(ts, quantised, where="mid", color="#d93025", lw=2)
for q in np.linspace(0, 1, levels):
    axes[2].axhline(q, color="#e8eaed", lw=0.8, zorder=0)
axes[2].set_title(f"quantisation: round to {levels} levels (3 bits)")
for ax in axes:
    ax.set_xlabel("position x"); ax.grid(alpha=0.2)
axes[0].set_ylabel("intensity")
plt.suptitle("Figure 2.1 - Sampling and quantisation of a 1D brightness profile", fontweight="bold")
plt.tight_layout(); plt.show()

**Figure 2.2 — Zooming into an image until the numbers appear.** Every digital image is ultimately a table of integers.

In [ ]:
cam = skimage.data.camera()          # 512x512 greyscale, public domain
r0, c0 = 180, 240
fig, axes = plt.subplots(1, 3, figsize=(17, 5.6))
axes[0].imshow(cam, cmap="gray"); axes[0].add_patch(plt.Rectangle((c0, r0), 40, 40, fill=False, ec="#fbbc04", lw=2))
axes[0].set_title("512 x 512 image")
patch = cam[r0:r0 + 40, c0:c0 + 40]
axes[1].imshow(patch, cmap="gray"); axes[1].add_patch(plt.Rectangle((14.5, 14.5), 10, 10, fill=False, ec="#d93025", lw=2))
axes[1].set_title("40 x 40 zoom: individual pixels visible")
tiny = patch[15:25, 15:25]
axes[2].imshow(tiny, cmap="gray", vmin=0, vmax=255)
for (i, j), v in np.ndenumerate(tiny):
    axes[2].text(j, i, str(v), ha="center", va="center", fontsize=8, color="w" if v < 128 else "k")
axes[2].set_title("10 x 10 pixels: the numbers the computer sees")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 3. Images are NumPy arrays

| Image type | NumPy shape | Typical `dtype` | Value range |
|---|---|---|---|
| greyscale | `(H, W)` | `uint8` | 0–255 |
| colour (OpenCV) | `(H, W, 3)`, channel order **B, G, R** | `uint8` | 0–255 |
| industrial 12-bit camera | `(H, W)` | `uint16` | 0–4095 |
| after processing | any | `float32` | often 0–1 |

**Coordinates.** NumPy indexes `img[row, column]` = `img[y, x]`, with the origin at the **top-left** corner and *y pointing down*.
OpenCV drawing functions take points as `(x, y)`. Mixing the two up is the most common beginner bug.

In [ ]:
img = load_sample("smarties.png")        # OpenCV sample image (BGR)
print("type :", type(img))
print("shape:", img.shape, "-> H, W, C")
print("dtype:", img.dtype, "| min", img.min(), "| max", img.max())
print("pixel at row 100, column 200 (B, G, R):", img[100, 200])

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].annotate("", xy=(150, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#d93025", lw=3))
axes[0].annotate("", xy=(0, 120), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="#1a73e8", lw=3))
axes[0].text(155, 12, "x = column (img[:, x])", color="#d93025", fontweight="bold")
axes[0].text(8, 130, "y = row\n(img[y, :])", color="#1a73e8", fontweight="bold")
axes[0].plot(200, 100, "o", ms=12, mfc="none", mec="#fbbc04", mew=3)
axes[0].text(210, 95, "img[100, 200]", color="#fbbc04", fontweight="bold")
axes[0].set_title("Image coordinates: origin top-left, y points DOWN")

ax = axes[1]; ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 7)
for k, (name, color) in enumerate([("R (index 2)", "#f28b82"), ("G (index 1)", "#81c995"), ("B (index 0)", "#8ab4f8")]):
    ax.add_patch(plt.Rectangle((2 + 0.5 * k, 1 + 0.5 * k), 5, 4, fc=color, ec="k", alpha=0.9))
    ax.text(7.2 + 0.5 * k, 1.2 + 0.5 * k, name, fontsize=10)
ax.text(4.5, 0.4, "W = 413 columns", ha="center"); ax.text(1.4, 3, "H = 356\nrows", ha="center")
ax.set_title("Memory layout: H x W x 3 (OpenCV order B, G, R)")
plt.suptitle("Figure 3.1 - Coordinates and channels", fontweight="bold")
plt.tight_layout(); plt.show()

**Regions of interest (ROIs)** are simply slices. A slice is a *view*: changing it changes the original image unless you `.copy()` it.

In [ ]:
roi = img[200:330, 50:250]                 # rows 200-329, columns 50-249
view_demo = img.copy()
view_demo[200:330, 50:250] = 255 - view_demo[200:330, 50:250]   # invert only the ROI
b, g, r = cv2.split(img)
show(roi, view_demo, b, g, r, titles=["ROI img[200:330, 50:250]", "ROI inverted in place", "Blue channel", "Green channel",
                                     "Red channel"], figsize=(22, 4))

## 4. Sampling, resolution and aliasing

**Resolution** is the number of samples. Fewer samples lose fine detail. Worse, detail finer than two pixels does not simply
disappear: it turns into false, coarser patterns called **aliasing** (moiré). The **Nyquist–Shannon sampling theorem** says a pattern
must be sampled at least twice per period to be represented faithfully. Cameras use an optical anti-aliasing filter, and software
should **blur before downsampling** (`cv2.INTER_AREA` or a Gaussian pyramid, Week 5).

In [ ]:
factors = [1, 4, 8, 16, 32]
fig, axes = plt.subplots(1, len(factors), figsize=(19, 4))
for ax, f in zip(axes, factors):
    small = cv2.resize(cam, (512 // f, 512 // f), interpolation=cv2.INTER_AREA)
    ax.imshow(cv2.resize(small, (512, 512), interpolation=cv2.INTER_NEAREST), cmap="gray")
    ax.set_title(f"{512 // f} x {512 // f} pixels"); ax.axis("off")
plt.suptitle("Figure 4.1 - Spatial resolution: the same scene with fewer and fewer samples", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# A "zone plate": concentric rings whose frequency increases towards the edge
yy, xx = np.mgrid[-1:1:1024j, -1:1:1024j]
zone = (0.5 + 0.5 * np.cos(180 * (xx ** 2 + yy ** 2))).astype(np.float32)
naive = zone[::8, ::8]                                            # keep every 8th pixel: no anti-aliasing
smooth = cv2.resize(cv2.GaussianBlur(zone, (0, 0), 4), (128, 128), interpolation=cv2.INTER_AREA)
fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
for ax, im, title in zip(axes, [zone, naive, smooth], ["original 1024 x 1024 (fine rings at the edge)",
                                                       "every 8th pixel: false rings (ALIASING)",
                                                       "blur first, then downsample: smooth grey"]):
    ax.imshow(im, cmap="gray", interpolation="nearest"); ax.set_title(title); ax.axis("off")
plt.suptitle("Figure 4.2 - Aliasing: undersampled detail turns into fake patterns", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: resolution
Downsample and then enlarge again with nearest-neighbour interpolation. Compare `INTER_NEAREST` (aliasing) with `INTER_AREA` (anti-aliased).

In [ ]:
def resolution_demo(keep_percent=25, method="INTER_AREA"):
    h, w = img.shape[:2]
    s = keep_percent / 100
    small = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=getattr(cv2, method))
    big = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    show(img, big, titles=["original", f"{small.shape[1]} x {small.shape[0]} ({method}), enlarged"], figsize=(12, 5))


interact(resolution_demo, keep_percent=widgets.IntSlider(value=25, min=2, max=100, step=1),
         method=["INTER_AREA", "INTER_NEAREST", "INTER_LINEAR"])

## 5. Quantisation and bit depth

With $b$ bits a pixel can take $2^b$ values. Too few levels cause **false contours**: smooth gradients turn into visible steps.
Industrial cameras often deliver 10–12 bits (1,024–4,096 levels) for precise measurement, while displays use 8 bits.

| Bits | Levels | Typical use |
|---|---|---|
| 1 | 2 | binary masks (black/white) |
| 8 | 256 | most images, displays, JPEG |
| 12 | 4,096 | machine-vision and scientific cameras |
| 16 | 65,536 | depth maps, HDR, medical |

In [ ]:
def quantise_demo_img(image, bits):
    step = 256 // (2 ** bits)
    return (image // step) * step + step // 2


fig, axes = plt.subplots(2, 5, figsize=(19, 7.5), gridspec_kw={"height_ratios": [3, 1.3]})
for j, bits in enumerate([8, 5, 3, 2, 1]):
    q = quantise_demo_img(cam, bits)
    axes[0, j].imshow(q, cmap="gray", vmin=0, vmax=255); axes[0, j].axis("off")
    axes[0, j].set_title(f"{bits} bit{'s' if bits > 1 else ''} = {2 ** bits} levels")
    axes[1, j].hist(q.ravel(), bins=256, range=(0, 256), color="#1a73e8")
    axes[1, j].set_yticks([])
plt.suptitle("Figure 5.1 - Bit depth: fewer levels cause false contours (top) and a sparse histogram (bottom)", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: bit depth on a smooth gradient

In [ ]:
gradient = np.tile(np.linspace(0, 255, 600).astype(np.uint8), (80, 1))


def bits_demo(bits=4):
    q = quantise_demo_img(gradient, bits)
    fig, axes = plt.subplots(2, 1, figsize=(11, 3.4))
    axes[0].imshow(gradient, cmap="gray", vmin=0, vmax=255, aspect="auto"); axes[0].set_title("8-bit gradient")
    axes[1].imshow(q, cmap="gray", vmin=0, vmax=255, aspect="auto"); axes[1].set_title(f"{bits}-bit: {2 ** bits} levels")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout(); plt.show()


interact(bits_demo, bits=widgets.IntSlider(value=4, min=1, max=8))

### An image is also a surface
Treating brightness as height helps you think about gradients (Week 3), edges (Week 4) and features (Week 5). Rotate the plot with your mouse.

In [ ]:
small = cv2.resize(cam, (128, 128), interpolation=cv2.INTER_AREA)
fig = go.Figure(go.Surface(z=small[::-1].astype(float), colorscale="gray", showscale=False))
fig.update_layout(title="Figure 5.2 - The camera-man image as a brightness surface (interactive)", height=520,
                  scene=dict(xaxis_title="x (column)", yaxis_title="y (row)", zaxis_title="brightness",
                             aspectratio=dict(x=1, y=1, z=0.4)))
fig.show()

## 6. Data types: the `uint8` trap

`uint8` values live in 0–255. NumPy arithmetic **wraps around** (250 + 10 = 4), while OpenCV functions **saturate**
(clip at 255). Convert to `float32` for calculations, then back to `uint8` for display and saving.

In [ ]:
a = np.array([250], dtype=np.uint8)
print("NumPy  : 250 + 10 =", (a + np.uint8(10))[0], "   <- wrapped around!")
print("OpenCV : 250 + 10 =", cv2.add(a, 10).ravel()[0], "  <- saturated at 255")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
wrong = gray + np.uint8(80)
right = cv2.add(gray, 80)
via_float = np.clip(gray.astype(np.float32) + 80, 0, 255).astype(np.uint8)
show(gray, wrong, right, via_float, titles=["original", "NumPy +80: bright pixels wrap to black!", "cv2.add(+80): saturates",
                                          "float, clip, back to uint8"], figsize=(20, 4.5))

## 7. Basic operations: resize, flip, rotate, blend

When you enlarge an image, new pixel values must be **interpolated** from their neighbours:

| Method | OpenCV flag | Quality | Speed | Use for |
|---|---|---|---|---|
| nearest neighbour | `INTER_NEAREST` | blocky | fastest | label masks (never mix class IDs) |
| bilinear | `INTER_LINEAR` | smooth | fast | default for enlarging |
| bicubic | `INTER_CUBIC` | sharper | slower | high-quality enlarging |
| Lanczos | `INTER_LANCZOS4` | sharpest | slowest | photo editing |
| area | `INTER_AREA` | anti-aliased | fast | **shrinking** |

In [ ]:
crop = img[235:275, 150:200]     # edge of a dark candy
methods = ["INTER_NEAREST", "INTER_LINEAR", "INTER_CUBIC", "INTER_LANCZOS4"]
show(*[cv2.resize(crop, None, fx=8, fy=8, interpolation=getattr(cv2, m)) for m in methods],
     titles=[f"x8 {m}" for m in methods], figsize=(20, 4.5))
flips = [cv2.flip(img, 1), cv2.flip(img, 0), cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE),
         cv2.addWeighted(img, 0.6, cv2.flip(img, 1), 0.4, 0)]
show(*flips, titles=["flip horizontal", "flip vertical", "rotate 90 deg", "blend 60/40 (addWeighted)"], figsize=(20, 4))

### BGR vs RGB
OpenCV stores colour as **B, G, R**. Matplotlib, PIL, scikit-image and PyTorch expect **R, G, B**. Forgetting to convert swaps red
and blue, which is harmless for display but catastrophic for a colour-sorting machine.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].imshow(img); axes[0].set_title("BGR array shown as RGB (WRONG: red and blue swapped)")
axes[1].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[1].set_title("after cv2.cvtColor(img, COLOR_BGR2RGB)")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Speed: vectorise, never loop over pixels
A 1-megapixel image has a million pixels. Python loops are ~100–1000× slower than NumPy or OpenCV, which run optimised C code.

In [ ]:
big = cv2.resize(gray, (1000, 1000))
n = 200 if SMOKE else 1000
t0 = time.perf_counter()
out = np.empty_like(big[:n])
for r in range(n):
    for c in range(big.shape[1]):
        out[r, c] = 255 - big[r, c]
t_loop = (time.perf_counter() - t0) * big.shape[0] / n     # scaled to the full image
t0 = time.perf_counter(); _ = 255 - big; t_numpy = time.perf_counter() - t0
t0 = time.perf_counter(); _ = cv2.bitwise_not(big); t_cv = time.perf_counter() - t0
times = {"Python loops": t_loop, "NumPy (255 - img)": t_numpy, "OpenCV bitwise_not": t_cv}
fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(list(times), [v * 1000 for v in times.values()], color=["#d93025", "#1a73e8", "#188038"])
ax.set_xscale("log"); ax.set_xlabel("milliseconds to invert a 1000 x 1000 image (log scale)")
for i, v in enumerate(times.values()):
    ax.text(v * 1000 * 1.2, i, f"{v * 1000:.2f} ms", va="center")
ax.set_title("Figure 7.1 - Vectorised code is orders of magnitude faster", fontweight="bold")
plt.show()

## 8. File formats and compression

**JPEG** throws information away (lossy) to make files small. Its 8×8-block artefacts can look like defects, or hide real ones.
For inspection and measurement, store **lossless** formats (PNG, TIFF, BMP), or raw camera data.

In [ ]:
results = {}
for name, params in [("PNG (lossless)", (".png", [])), ("JPEG q=95", (".jpg", [cv2.IMWRITE_JPEG_QUALITY, 95])),
                     ("JPEG q=50", (".jpg", [cv2.IMWRITE_JPEG_QUALITY, 50])), ("JPEG q=10", (".jpg", [cv2.IMWRITE_JPEG_QUALITY, 10]))]:
    ok, buf = cv2.imencode(params[0], img, params[1])
    dec = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    results[name] = (len(buf) / 1024, dec, np.abs(dec.astype(int) - img.astype(int)).mean())
fig, axes = plt.subplots(2, 4, figsize=(19, 7.5))
for j, (name, (kb, dec, err)) in enumerate(results.items()):
    axes[0, j].imshow(cv2.cvtColor(dec[220:300, 100:220], cv2.COLOR_BGR2RGB), interpolation="nearest")
    axes[0, j].set_title(f"{name}: {kb:.0f} KB")
    diff = np.abs(dec.astype(int) - img.astype(int)).sum(2)[220:300, 100:220]
    axes[1, j].imshow(diff, cmap="magma", vmin=0, vmax=60); axes[1, j].set_title(f"error (mean {err:.2f})")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Figure 8.1 - Compression: file size (top) vs introduced error (bottom), zoomed", fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Case study: is the part present?

The simplest industrial vision check measures the **mean brightness inside a fixed ROI**. We simulate a conveyor camera where a
bright part should sit inside the green inspection window. Because the lighting and the part position are controlled, a single
threshold is enough. This is why good **fixturing and lighting** (Week 2) make algorithms simple.

In [ ]:
rng = np.random.default_rng(0)


def conveyor_frame(part_present, shift=0, light=1.0):
    frame = np.full((240, 320), 60, np.uint8)                      # dark belt
    frame[:, ::40] = 45                                            # belt seams
    if part_present:
        cv2.circle(frame, (160 + shift, 120), 45, 200, -1)          # bright part
        cv2.circle(frame, (160 + shift, 120), 12, 90, -1)           # centre hole
    frame = np.clip(frame * light + rng.normal(0, 6, frame.shape), 0, 255).astype(np.uint8)
    return frame


ROI = (slice(80, 160), slice(120, 200))            # rows, columns of the inspection window
THRESHOLD = 110
frames = [conveyor_frame(True), conveyor_frame(False), conveyor_frame(True, shift=25),
          conveyor_frame(True, light=0.55), conveyor_frame(False, light=1.4)]
names = ["part OK", "part missing", "part shifted 25 px", "lamp ageing (55%)", "missing + bright lamp"]
fig, axes = plt.subplots(1, 5, figsize=(20, 4.2))
for ax, fr, name in zip(axes, frames, names):
    mean = fr[ROI].mean()
    present = mean > THRESHOLD
    vis = cv2.cvtColor(fr, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(vis, (120, 80), (199, 159), (0, 200, 0) if present else (220, 0, 0), 2)
    ax.imshow(vis); ax.axis("off")
    ax.set_title(f"{name}\nROI mean {mean:.0f} -> {'PRESENT' if present else 'MISSING'}",
                 color="#188038" if present else "#d93025")
plt.suptitle(f"Figure 9.1 - A one-line inspection rule (ROI mean > {THRESHOLD}) and how it can fail", fontweight="bold")
plt.tight_layout(); plt.show()

**Discussion.** The rule works for the nominal cases, but a *shifted part* and *changed lighting* move the ROI mean close to or across
the threshold. Robust systems fix this with (a) mechanical fixturing and stable lighting, (b) relative measurements (compare with a
reference image), and (c) smarter algorithms: segmentation (Week 4), features (Week 5) and learning (Week 6).

## 10. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check cell. The checks test *behaviour*, not a particular solution.

### E1. Greyscale by hand
Implement greyscale conversion **without** `cv2.cvtColor`, using the ITU-R BT.601 weights
$Y = 0.299R + 0.587G + 0.114B$. Remember the input is **BGR** and the output must be `uint8`.

In [ ]:
def to_gray(img_bgr):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    g = to_gray(img)
    assert isinstance(g, np.ndarray) and g.shape == img.shape[:2], f"expected shape {img.shape[:2]}, got {getattr(g, 'shape', None)}"
    assert g.dtype == np.uint8, f"expected dtype uint8, got {g.dtype}"
    ref = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(int)
    err = np.abs(g.astype(int) - ref).max()
    assert err <= 1, f"differs from OpenCV by up to {err} grey levels: check the channel order (BGR!)"


check("E1 to_gray", _test_e1)

### E2. Re-quantise an image
Return a `uint8` image with only `2**bits` distinct grey levels, spread over 0–255, with the smallest possible rounding error.

In [ ]:
def quantise(gray_img, bits):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    for bits in (1, 2, 4, 6):
        q = quantise(cam, bits)
        assert q.dtype == np.uint8 and q.shape == cam.shape, "keep the shape and return uint8"
        n = len(np.unique(q))
        assert n <= 2 ** bits, f"{bits} bits allow at most {2 ** bits} levels, you produced {n}"
        err = np.abs(q.astype(int) - cam.astype(int)).max()
        assert err <= 256 // 2 ** bits, f"{bits} bits: max error {err} is larger than half a step ({256 // 2 ** bits})"


check("E2 quantise", _test_e2)

### E3. Safe brightness change
Add `delta` (which may be negative) to a `uint8` image **without wrap-around**, using only NumPy (no `cv2.add`).

In [ ]:
def adjust_brightness(gray_img, delta):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    for delta in (100, -100, 0, 255):
        out = adjust_brightness(gray, delta)
        assert out.dtype == np.uint8, "return uint8"
        ref = np.clip(gray.astype(int) + delta, 0, 255).astype(np.uint8)
        assert np.array_equal(out, ref), f"delta={delta}: values wrapped around or were not clipped to 0-255"


check("E3 adjust_brightness", _test_e3)

### E4. Make the presence check robust (no self-check)
Change the Section 9 rule so that it still gives the correct answer for all five frames. Ideas: normalise the ROI mean by the
mean brightness *outside* the ROI (lamp changes affect both), or enlarge the ROI. Explain your choice in two sentences.

In [ ]:
# E4: your code here

## 11. Challenge: characterise your own camera

Use your laptop webcam (set `RUN_WEBCAM = True`, locally only) to measure:
1. the **resolution** and **frame rate** it actually delivers;
2. the **temporal noise**: capture 30 frames of a static scene and plot the per-pixel standard deviation as an image. Where is the noise highest, in dark or bright areas? Why?
3. the effect of **JPEG compression** on the noise image (save and reload one frame at quality 30).

In [ ]:
RUN_WEBCAM = False

if RUN_WEBCAM:
    cap = cv2.VideoCapture(0)
    frames, t0 = [], time.time()
    for _ in range(30):
        ok_cam, frame = cap.read()
        if ok_cam:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32))
    cap.release()
    fps = len(frames) / (time.time() - t0)
    stack = np.stack(frames)
    print(f"{stack.shape[2]} x {stack.shape[1]} pixels at about {fps:.1f} frames/s")
    show(stack.mean(0).astype(np.uint8), stack.std(0), titles=["mean of 30 frames", "per-pixel noise (std)"], cmap="magma")

## 12. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'A colour image loaded with cv2.imread has shape (480, 640, 3). What is its width?',
        "options": [
            '480 pixels',
            '640 pixels',
            '3 pixels',
            '1,920 pixels',
        ],
        "answer": '496b032d05',
        "explain": 'The shape is (height, width, channels): 480 rows and 640 columns.',
    },
    {
        "q": 'How many grey levels can a 10-bit camera represent?',
        "options": [
            '10',
            '100',
            '1,024',
            '65,536',
        ],
        "answer": '47a9a1defc',
        "explain": '2^10 = 1,024.',
    },
    {
        "q": 'What does np.uint8(250) + np.uint8(10) give in NumPy?',
        "options": [
            '260',
            '255',
            '4',
            'an error',
        ],
        "answer": 'f43cd57647',
        "explain": "uint8 arithmetic wraps around modulo 256: 260 - 256 = 4. OpenCV's cv2.add would saturate at 255.",
    },
    {
        "q": 'Fine stripes in a scene appear as coarse, wavy false patterns after downsampling. What is this called?',
        "options": [
            'Quantisation',
            'Aliasing',
            'Compression',
            'Saturation',
        ],
        "answer": '9b2f94b8ea',
        "explain": 'Detail finer than the sampling rate (Nyquist limit) folds into false low-frequency patterns. Blur before downsampling.',
    },
    {
        "q": 'Which interpolation should you use to resize a segmentation mask containing class IDs?',
        "options": [
            'INTER_NEAREST',
            'INTER_LINEAR',
            'INTER_CUBIC',
            'INTER_LANCZOS4',
        ],
        "answer": '09250bc18a',
        "explain": 'Other methods average neighbouring IDs and create class numbers that do not exist.',
    },
    {
        "q": 'Why is JPEG a poor choice for storing images used for defect inspection?',
        "options": [
            'It cannot store colour',
            'It is lossy: block artefacts can hide or imitate defects',
            'Files are too large',
            'OpenCV cannot read it',
        ],
        "answer": 'c3d670688a',
        "explain": 'Use lossless PNG/TIFF or raw data for measurement and inspection.',
    },
    {
        "q": 'In NumPy, img[50, 120] refers to...',
        "options": [
            'x = 50, y = 120',
            'row 50, column 120',
            'channel 50, pixel 120',
            'the 50th image in a batch',
        ],
        "answer": 'c171faadc2',
        "explain": 'NumPy indexes [row, column] = [y, x]. OpenCV drawing functions use (x, y) points.',
    },
], title='Week 01A quiz')

## 13. Exit ticket (reflection)
1. Name one inspection task where **higher resolution** matters more than **higher bit depth**, and one where it is the other way round.
2. The Section 9 check failed when the lamp aged. List two *hardware* and one *software* fix.
3. Why do machine-vision engineers say "the best algorithm is good lighting"?

## 14. What's new (2025–26)

> - **Global-shutter CMOS** sensors are now affordable in industrial cameras, avoiding the "rolling shutter" skew of phone cameras on fast conveyors.
> - **Event cameras** report per-pixel brightness *changes* with microsecond latency instead of frames: ideal for very fast motion (Week 10).
> - **Smart cameras and edge AI modules** (e.g. NVIDIA Jetson Orin, Raspberry Pi AI camera) run deep networks inside the camera, so decisions happen at the edge (Week 9).
> - OpenCV **5.x** is now on PyPI; everything in this course uses APIs that work in both OpenCV 4.x and 5.x.

## 15. References and further exploration

**Textbooks**
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §2.3 (the digital camera). [szeliski.org/Book](https://szeliski.org/Book/)
- Gonzalez, R. C. & Woods, R. E. (2018). *Digital Image Processing*, 4th ed., ch. 2 (sampling and quantisation). Pearson.
- Steger, C., Ulrich, M. & Wiedemann, C. (2018). *Machine Vision Algorithms and Applications*, 2nd ed., ch. 1–2. Wiley.
- Torralba, Isola & Freeman (2024). *Foundations of Computer Vision*, ch. 5 (imaging). [visionbook.mit.edu](https://visionbook.mit.edu/)

**Standards and classic papers**
- Shannon, C. E. (1949). *Communication in the presence of noise.* Proc. IRE 37(1) (the sampling theorem). [doi:10.1109/JRPROC.1949.232969](https://doi.org/10.1109/JRPROC.1949.232969)
- EMVA 1288: the standard for measuring and comparing machine-vision camera performance. [emva.org](https://www.emva.org/standards-technology/emva-1288/)
- ITU-R BT.601: luma weights used for colour-to-greyscale conversion. [itu.int](https://www.itu.int/rec/R-REC-BT.601)

**Video and interactive**
- Nayar, S. *First Principles of Computer Vision*: "Image Formation" and "Image Sensing" lectures. [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/)
- OpenCV Python tutorials: *Core operations*. [docs.opencv.org](https://docs.opencv.org/4.x/d7/d16/tutorial_py_table_of_contents_core.html)
- NumPy indexing guide. [numpy.org](https://numpy.org/doc/stable/user/basics.indexing.html)
- scikit-image: *A crash course on NumPy for images*. [scikit-image.org](https://scikit-image.org/docs/stable/user_guide/numpy_images.html)

**Images:** `smarties.png` from the OpenCV sample data; `camera` from scikit-image (public domain). All other images are generated in code.

## 16. Key takeaways
- A machine-vision system = lighting + optics + sensor + processing + decision + actuation. **The image quality limits everything downstream.**
- A digital image is a **sampled** (resolution) and **quantised** (bit depth) grid of numbers: a NumPy array `(H, W[, C])`.
- Undersampling causes **aliasing**; too few levels cause **false contours**.
- Index as `img[row, col]`; OpenCV uses **BGR**; `uint8` arithmetic wraps around, so compute in float or use OpenCV's saturating functions.
- Vectorise; never loop over pixels. Store inspection images **losslessly**.

**Next session (01B):** colour: how cameras sense it, colour spaces (RGB, HSV, Lab), histograms and colour-based sorting.